In [1]:
import numpy as np
import pandas as pd

with open('Calib.csv') as f:
    lines = f.read().splitlines()

blocks = []
current = []
for line in lines:
    if line.strip().lower().startswith('voltage,'):
        if current:
            blocks.append(current)
        current = []
    else:
        if line.strip():
            current.append(line)
if current:
    blocks.append(current)

print(f"Found {len(blocks)} taps")

Found 3 taps


In [2]:
def parse_tap(block):
    rows = [[p.strip() for p in line.split(',')] for line in block]
    nums = [[float(''.join(c for c in x if c.isdigit() or c == '.')) for x in row] for row in rows]
    nums = np.array(nums)  # col0 = voltage, col1/col2 = raw_adc & adc_voltage in unknown order

    err_A = np.mean(np.abs(nums[:, 2] / 4095 * 3.3 - nums[:, 1]))  # col1=adc_voltage, col2=raw_adc
    err_B = np.mean(np.abs(nums[:, 1] / 4095 * 3.3 - nums[:, 2]))  # col1=raw_adc, col2=adc_voltage

    if err_A < err_B:
        raw_adc, adc_v = nums[:, 2], nums[:, 1]
    else:
        raw_adc, adc_v = nums[:, 1], nums[:, 2]
    voltage = nums[:, 0]
    return voltage, raw_adc, adc_v

In [3]:
tap_calib = {}
for i, block in enumerate(blocks, start=1):
    voltage, raw_adc, adc_v = parse_tap(block)
    slope, offset = np.polyfit(raw_adc, voltage, 1)
    predicted = np.polyval([slope, offset], raw_adc)
    max_err = np.max(np.abs(voltage - predicted))

    tap_calib[f'Tap{i}'] = {'slope': slope, 'offset': offset, 'max_error_V': max_err, 'n_points': len(raw_adc)}
    print(f"Tap{i}: slope={slope:.8f}  offset={offset:.6f}  max_error={max_err:.4f}V")

calib_df = pd.DataFrame(tap_calib).T
calib_df

Tap1: slope=0.00105757  offset=0.461278  max_error=0.0315V
Tap2: slope=0.00241651  offset=0.456195  max_error=0.0560V
Tap3: slope=0.00341155  offset=1.066423  max_error=0.4100V


,slope,offset,max_error_V,n_points
Tap1,0.001058,0.461278,0.031508,5.0
Tap2,0.002417,0.456195,0.055962,5.0
Tap3,0.003412,1.066423,0.410025,5.0
